In [ ]:
from google.colab import files
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

print("📁 Please upload your Excel or CSV file")
print("Expected columns: date, return, sentiment")
print("-" * 50)

uploaded = files.upload()
filename = list(uploaded.keys())[0]

# Load file
if filename.endswith('.csv'):
    df = pd.read_csv(filename)
else:
    df = pd.read_excel(filename)

# Clean and prepare
df['date'] = pd.to_datetime(df['date'])
df = df.sort_values('date').reset_index(drop=True)

# Remove non-trading days (return = 0)
df = df[df['return'] != 0].reset_index(drop=True)

print(f"\n✅ Loaded {len(df)} observations")
print(f"   Date range: {df['date'].min()} to {df['date'].max()}")
print(f"\nFirst 5 rows:")
print(df.head())

📁 Please upload your Excel or CSV file
Expected columns: date, return, sentiment
--------------------------------------------------


Saving dataset_title_only_full.xlsx to dataset_title_only_full (1).xlsx

✅ Loaded 506 observations
   Date range: 2024-01-02 00:00:00 to 2026-01-07 00:00:00

First 5 rows:
        date    return  sentiment
0 2024-01-02 -0.039506  -0.032163
1 2024-01-03 -0.002874  -0.032163
2 2024-01-04  0.012733  -0.032163
3 2024-01-05  0.013160  -0.032163
4 2024-01-08  0.041465  -0.032163


In [ ]:
import statsmodels.api as sm
from statsmodels.tsa.arima.model import ARIMA
from scipy import stats
import numpy as np

print("=" * 60)
print("ARIMAX MODEL FIT TEST")
print("=" * 60)

y = df['return'].values
sentiment = df['sentiment'].values

# Fit ARIMAX(1,0,1) with sentiment as exogenous variable
model = ARIMA(y, order=(1, 0, 1), exog=sentiment)
fitted = model.fit()

# Null model (intercept only, no AR/MA, no sentiment)
null_model = ARIMA(y, order=(0, 0, 0))
null_fitted = null_model.fit()

# Calculate SSR (Sum of Squared Residuals) manually from residuals
null_ssr = np.sum(null_fitted.resid ** 2)
full_ssr = np.sum(fitted.resid ** 2)

# F-test
n = len(y)
k = len(fitted.params)
f_stat = ((null_ssr - full_ssr) / (k - 1)) / (full_ssr / (n - k))
f_pvalue = 1 - stats.f.cdf(f_stat, k-1, n-k)

# Also get R-squared
r_squared = 1 - (full_ssr / null_ssr)

print(f"\nNull model SSR: {null_ssr:.6f}")
print(f"Full model SSR: {full_ssr:.6f}")
print(f"F-statistic: {f_stat:.4f}")
print(f"p-value: {f_pvalue:.6f}")
print(f"R-squared: {r_squared:.4f}")
print("-" * 60)

if f_pvalue < 0.05:
    print("✅ CONCLUSION: ARIMAX model is a GOOD FIT")
    print("   (Model with sentiment is significantly better than null model)")
else:
    print("❌ CONCLUSION: ARIMAX model is NOT a good fit")
    print("   (Sentiment does not add significant explanatory power)")

# Also show the sentiment coefficient and its p-value
print("\n" + "-" * 60)
print("SENTIMENT COEFFICIENT DETAILS:")
print("-" * 60)

# Find which parameter is sentiment
for i, name in enumerate(fitted.param_names):
    if 'x' in name.lower():
        print(f"Sentiment coefficient: {fitted.params[i]:.6f}")
        print(f"Sentiment p-value: {fitted.pvalues[i]:.6f}")

        if fitted.pvalues[i] < 0.05:
            print("✓ Sentiment coefficient is statistically significant (p < 0.05)")
        else:
            print("✗ Sentiment coefficient is NOT statistically significant (p ≥ 0.05)")
        break

ARIMAX MODEL FIT TEST

Null model SSR: 0.362016
Full model SSR: 0.361027
F-statistic: 0.3430
p-value: 0.848905
R-squared: 0.0027
------------------------------------------------------------
❌ CONCLUSION: ARIMAX model is NOT a good fit
   (Sentiment does not add significant explanatory power)

------------------------------------------------------------
SENTIMENT COEFFICIENT DETAILS:
------------------------------------------------------------
Sentiment coefficient: 0.003035
Sentiment p-value: 0.692162
✗ Sentiment coefficient is NOT statistically significant (p ≥ 0.05)


In [ ]:
import xgboost as xgb
from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import r2_score

print("=" * 60)
print("XGBOOST + OPTUNA MODEL FIT TEST")
print("=" * 60)

# Create simple lag features (from your paper's structure)
df_xgb = df.copy()
df_xgb['return_lag1'] = df_xgb['return'].shift(1)
df_xgb['return_lag2'] = df_xgb['return'].shift(2)
df_xgb['sentiment_lag1'] = df_xgb['sentiment'].shift(1)
df_xgb['sentiment_lag2'] = df_xgb['sentiment'].shift(2)
df_xgb = df_xgb.dropna().reset_index(drop=True)

# Features and target
feature_cols = ['return_lag1', 'return_lag2', 'sentiment', 'sentiment_lag1', 'sentiment_lag2']
X = df_xgb[feature_cols].values
y = df_xgb['return'].values

# Use Optuna's best parameters from your paper (page 100)
best_params = {
    'n_estimators': 281,
    'max_depth': 7,
    'learning_rate': 0.000145,
    'subsample': 0.804,
    'colsample_bytree': 0.585,
    'reg_alpha': 0.390,
    'reg_lambda': 0.531,
    'min_child_weight': 1,
    'random_state': 42
}

model = xgb.XGBRegressor(**best_params)

# Time Series Cross-Validation (3 folds)
tscv = TimeSeriesSplit(n_splits=3)
r2_scores = []

for train_idx, val_idx in tscv.split(X):
    X_train, X_val = X[train_idx], X[val_idx]
    y_train, y_val = y[train_idx], y[val_idx]

    model.fit(X_train, y_train, verbose=False)
    y_pred = model.predict(X_val)
    r2_scores.append(r2_score(y_val, y_pred))

mean_r2 = np.mean(r2_scores)

print(f"\nCross-validated R² scores: {[round(s, 4) for s in r2_scores]}")
print(f"Mean R²: {mean_r2:.4f}")
print("-" * 60)

if mean_r2 > 0:
    print("✅ CONCLUSION: XGBoost model is a GOOD FIT")
    print("   (R² > 0 means model predicts better than simply guessing the mean)")
else:
    print("❌ CONCLUSION: XGBoost model is NOT a good fit")
    print("   (R² ≤ 0 means model performs no better than baseline)")

XGBOOST + OPTUNA MODEL FIT TEST

Cross-validated R² scores: [-0.0493, -0.0128, -0.0005]
Mean R²: -0.0209
------------------------------------------------------------
❌ CONCLUSION: XGBoost model is NOT a good fit
   (R² ≤ 0 means model performs no better than baseline)


In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout, Bidirectional
from tensorflow.keras.optimizers import Adam
from sklearn.preprocessing import MinMaxScaler

print("=" * 60)
print("GA-LSTM MODEL FIT TEST")
print("=" * 60)

# Prepare sequence data (lookback = 5 from your paper)
def create_sequences(data, lookback=5):
    X, y = [], []
    for i in range(lookback, len(data)):
        X.append(data[i-lookback:i])
        y.append(data[i])
    return np.array(X), np.array(y)

# Scale returns
scaler = MinMaxScaler()
returns_scaled = scaler.fit_transform(df[['return']].values)

X, y = create_sequences(returns_scaled.flatten(), lookback=5)

# Split (80/20 sequential)
split = int(len(X) * 0.8)
X_train, X_val = X[:split], X[split:]
y_train, y_val = y[:split], y[split:]

# Reshape for LSTM: (samples, timesteps, features)
X_train = X_train.reshape(X_train.shape[0], X_train.shape[1], 1)
X_val = X_val.reshape(X_val.shape[0], X_val.shape[1], 1)

# GA-LSTM architecture from your paper (page 94-95)
model = Sequential([
    Bidirectional(LSTM(128, return_sequences=True), input_shape=(5, 1)),
    Dropout(0.3744),
    LSTM(64),
    Dropout(0.3744),
    Dense(1)
])

model.compile(optimizer=Adam(learning_rate=0.005015), loss='mse')

# Train
history = model.fit(
    X_train, y_train,
    epochs=50,
    batch_size=32,
    validation_data=(X_val, y_val),
    verbose=0
)

# Extract losses
train_loss = history.history['loss'][-1]
val_loss = history.history['val_loss'][-1]
min_val_loss = min(history.history['val_loss'])
min_val_epoch = history.history['val_loss'].index(min_val_loss)

print(f"\nFinal Training Loss: {train_loss:.6f}")
print(f"Final Validation Loss: {val_loss:.6f}")
print(f"Best Validation Loss: {min_val_loss:.6f} (at epoch {min_val_epoch+1})")
print("-" * 60)

# Fit criteria from your paper
if val_loss <= min_val_loss * 1.1:
    print("✅ CONCLUSION: GA-LSTM model is a GOOD FIT")
    print("   (Validation loss stable, no significant overfitting)")
elif val_loss <= min_val_loss * 1.3:
    print("⚠️ CONCLUSION: GA-LSTM fit is ACCEPTABLE")
    print("   (Mild overfitting but may still generalize)")
else:
    print("❌ CONCLUSION: GA-LSTM model is NOT a good fit")
    print("   (Validation loss increased significantly → overfitting)")

GA-LSTM MODEL FIT TEST

Final Training Loss: 0.006177
Final Validation Loss: 0.008042
Best Validation Loss: 0.007833 (at epoch 44)
------------------------------------------------------------
✅ CONCLUSION: GA-LSTM model is a GOOD FIT
   (Validation loss stable, no significant overfitting)
